# MAgent2 MF-DSRQ Torch Evaluation

Evaluate trained MF-DSRQ torch robust-epsilon checkpoints against PyTorch MFRL baselines on `battle_v4`. Each epsilon cell compares MF-DSRQ against IQL, AC, and MFQ with both red/blue side assignments.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import torch


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "discrete_action_space").is_dir() and (path / "requirements.txt").exists():
            return path
    raise RuntimeError("Could not find the SRE-DQN repo root from the current notebook directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.mean_field_dsrq.notebook_utils import (
    evaluate_mfdsrq_torch_epsilon_against_baselines,
    find_latest_mfrl_run,
    load_mfdsrq_config,
    mfdsrq_epsilon_config,
    plot_mfdsrq_torch_baseline_bars,
)

MAP_SIZE = 40
MAX_CYCLES = 400
SEED = 42
ROBUST_DISTANCE = "tv"
ROBUST_LP_FALLBACK = "greedy_tv"
ROBUST_POLICY_TEMPERATURE = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EVAL_EPISODES_PER_SIDE = 25
EVALUATE_BOTH_SIDES = True
EVAL_EPSILONS = [0.01, 0.1, 0.5]
BASELINE_ALGORITHMS = ("iql", "ac", "mfq")
EVAL_WORKERS = 1 if DEVICE == "cuda" else min(6, os.cpu_count() or 1)
EVAL_EPISODE_CHUNK_SIZE = 100
EVAL_SHOW_PROGRESS = True
EVAL_MAX_STEPS = MAX_CYCLES

MFDSRQ_OUTPUT_ROOT = ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "mf_srq_torch_epsilon_training"
BASELINE_ROOT = ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "mfrl_baselines"

MFDSRQ_CHECKPOINT_PATHS = {
    0.01: {
        "main": MFDSRQ_OUTPUT_ROOT / "eps_0_01" / "battle_v4" / "seed42" / "ckpt_main_best.pt",
        "opponent": MFDSRQ_OUTPUT_ROOT / "eps_0_01" / "battle_v4" / "seed42" / "ckpt_opponent_best.pt",
    },
    0.1: {
        "main": MFDSRQ_OUTPUT_ROOT / "eps_0_1" / "battle_v4" / "seed42" / "ckpt_main_best.pt",
        "opponent": MFDSRQ_OUTPUT_ROOT / "eps_0_1" / "battle_v4" / "seed42" / "ckpt_opponent_best.pt",
    },
    0.5: {
        "main": MFDSRQ_OUTPUT_ROOT / "eps_0_5" / "battle_v4" / "seed42" / "ckpt_main_best.pt",
        "opponent": MFDSRQ_OUTPUT_ROOT / "eps_0_5" / "battle_v4" / "seed42" / "ckpt_opponent_best.pt",
    },
}

BASE_CFG = load_mfdsrq_config(
    overrides={
        "algorithm": "mf_srq_torch",
        "env_name": "battle_v4",
        "env_backend": "magent2",
        "map_size": MAP_SIZE,
        "max_cycles": MAX_CYCLES,
        "output_dir": str(MFDSRQ_OUTPUT_ROOT),
        "seed": SEED,
        "device": DEVICE,
        "use_gpu": DEVICE == "cuda",
        "robust_distance": ROBUST_DISTANCE,
        "robust_lp_fallback": ROBUST_LP_FALLBACK,
        "robust_policy_temperature": ROBUST_POLICY_TEMPERATURE,
    },
)

BASELINE_RUN_DIRS = {
    algorithm: find_latest_mfrl_run(algorithm, BASELINE_ROOT)
    for algorithm in BASELINE_ALGORITHMS
}
COMPARISONS = {}


def evaluate_and_plot_epsilon(epsilon):
    cfg = mfdsrq_epsilon_config(BASE_CFG, epsilon, MFDSRQ_OUTPUT_ROOT)
    comparison = evaluate_mfdsrq_torch_epsilon_against_baselines(
        cfg,
        epsilon,
        MFDSRQ_CHECKPOINT_PATHS[epsilon],
        baseline_root=BASELINE_ROOT,
        algorithms=BASELINE_ALGORITHMS,
        baseline_folders=BASELINE_RUN_DIRS,
        num_episodes_per_side=EVAL_EPISODES_PER_SIDE,
        max_steps=EVAL_MAX_STEPS,
        evaluate_both_sides=EVALUATE_BOTH_SIDES,
        workers=EVAL_WORKERS,
        episode_chunk_size=EVAL_EPISODE_CHUNK_SIZE,
        show_progress=EVAL_SHOW_PROGRESS,
        save=True,
        device=DEVICE,
    )
    _ = plot_mfdsrq_torch_baseline_bars(comparison, epsilon=epsilon, save=True)
    return comparison

{
    "mfdsrq_output_root": str(MFDSRQ_OUTPUT_ROOT),
    "baseline_root": str(BASELINE_ROOT),
    "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
    "eval_epsilons": EVAL_EPSILONS,
    "eval_episodes_per_side": EVAL_EPISODES_PER_SIDE,
    "evaluate_both_sides": EVALUATE_BOTH_SIDES,
    "eval_workers": EVAL_WORKERS,
    "eval_episode_chunk_size": EVAL_EPISODE_CHUNK_SIZE,
    "eval_show_progress": EVAL_SHOW_PROGRESS,
    "checkpoint_paths": {epsilon: {team: str(path) for team, path in paths.items()} for epsilon, paths in MFDSRQ_CHECKPOINT_PATHS.items()},
}


## Baseline Checkpoints

In [ ]:
pd.DataFrame([
    {"algorithm": algorithm, "run_dir": str(run_dir)}
    for algorithm, run_dir in BASELINE_RUN_DIRS.items()
])


## MF-DSRQ Torch Epsilon 0.01

In [ ]:
comparison_eps_0_01 = evaluate_and_plot_epsilon(0.01)
COMPARISONS[0.01] = comparison_eps_0_01
pd.DataFrame(comparison_eps_0_01["rows"])


## MF-DSRQ Torch Epsilon 0.1

In [ ]:
comparison_eps_0_1 = evaluate_and_plot_epsilon(0.1)
COMPARISONS[0.1] = comparison_eps_0_1
pd.DataFrame(comparison_eps_0_1["rows"])


## MF-DSRQ Torch Epsilon 0.5

In [ ]:
comparison_eps_0_5 = evaluate_and_plot_epsilon(0.5)
COMPARISONS[0.5] = comparison_eps_0_5
pd.DataFrame(comparison_eps_0_5["rows"])


## Aggregate Table

In [ ]:
all_rows = []
for epsilon, comparison in COMPARISONS.items():
    for row in comparison["rows"]:
        all_rows.append({"epsilon": epsilon, **row})
pd.DataFrame(all_rows)
